# MDM Organization Manager

Interactive UI for viewing, searching, and editing organization data from `mdm_genai_assets.demoui.orgtable`.

**Features:**
- Opens in a **new browser window** for full-screen experience
- Smart search on organization name and ID
- Paginated data table with **Edit button per row**
- Edit modal with save-to-table and auto-refresh

**How to use:**
1. Run all cells below
2. Click the link that appears to open the UI in a new tab
3. Search, browse, and edit organization records

In [ ]:
# Cell 0: Install Dependencies
%pip install flask --quiet

In [ ]:
# Cell 1: Configuration & Imports
import json
import re
import threading
from datetime import datetime
from flask import Flask, request, jsonify, Response

# Table configuration
TABLE_NAME = "mdm_genai_assets.demoui.orgtable"
PAGE_SIZE = 25
FLASK_PORT = 5000

# Columns to display in the results table
DISPLAY_COLUMNS = [
    "organization_id",
    "organization_name",
    "organization_status",
    "bed_count",
    "source_system",
    "parent_organization_name",
    "genai_organization_name",
    "genai_primary_specialty",
]

# Columns that users can edit
EDITABLE_COLUMNS = [
    "organization_name",
    "organization_status",
    "bed_count",
    "is_subpart",
    "parent_organization_name",
    "genai_organization_name",
    "genai_primary_specialty",
    "genai_citation",
]

# All columns in the table
ALL_COLUMNS = [
    "organization_id", "organization_name", "organization_status",
    "organization_status_date", "bed_count", "is_subpart",
    "parent_organization_name", "enumeration_date", "last_update_date",
    "source_system", "source_record_id", "genai_organization_name",
    "genai_primary_specialty", "genai_citation", "_run_cycle",
    "_run_date", "_record_hash", "created_date", "updated_date",
]

# Status options
STATUS_OPTIONS = ["A", "I", "D"]

print(f"Configuration loaded. Table: {TABLE_NAME}, Port: {FLASK_PORT}")

In [ ]:
# Cell 2: Database Helper Functions

def sanitize_input(value):
    """Escape single quotes and special characters for safe SQL embedding."""
    if value is None:
        return None
    return str(value).replace("'", "''").replace("\\", "\\\\")


def build_search_where_clause(query):
    """Build WHERE clause for multi-token search across name and ID."""
    if not query or not query.strip():
        return ""
    tokens = query.strip().split()
    conditions = []
    for token in tokens:
        safe_token = sanitize_input(token.lower())
        conditions.append(
            f"(LOWER(organization_name) LIKE '%{safe_token}%' "
            f"OR LOWER(organization_id) LIKE '%{safe_token}%')"
        )
    return "WHERE " + " AND ".join(conditions)


def build_relevance_order(query):
    """Build ORDER BY clause with relevance scoring."""
    if not query or not query.strip():
        return "ORDER BY organization_name ASC"
    safe_query = sanitize_input(query.strip().lower())
    return (
        f"ORDER BY CASE "
        f"WHEN LOWER(organization_name) = '{safe_query}' THEN 0 "
        f"WHEN LOWER(organization_id) = '{safe_query}' THEN 1 "
        f"WHEN LOWER(organization_name) LIKE '{safe_query}%' THEN 2 "
        f"WHEN LOWER(organization_id) LIKE '{safe_query}%' THEN 3 "
        f"ELSE 4 END, organization_name ASC"
    )


def search_organizations(query="", page=0, page_size=PAGE_SIZE):
    """Search organizations with pagination. Returns (list_of_dicts, total_count)."""
    where_clause = build_search_where_clause(query)
    order_clause = build_relevance_order(query)
    offset = page * page_size

    count_sql = f"SELECT COUNT(*) as cnt FROM {TABLE_NAME} {where_clause}"
    total_count = spark.sql(count_sql).collect()[0]["cnt"]

    cols = ", ".join(ALL_COLUMNS)
    data_sql = (
        f"SELECT {cols} FROM {TABLE_NAME} "
        f"{where_clause} {order_clause} "
        f"LIMIT {page_size} OFFSET {offset}"
    )
    df = spark.sql(data_sql).toPandas()

    # Convert to list of dicts, handling NaN/NaT values
    rows = []
    for _, row in df.iterrows():
        d = {}
        for col in ALL_COLUMNS:
            val = row.get(col)
            if val is None or (hasattr(val, '__class__') and val.__class__.__name__ == 'NaTType'):
                d[col] = None
            elif hasattr(val, 'isoformat'):
                d[col] = val.isoformat()
            elif isinstance(val, float) and (val != val):  # NaN check
                d[col] = None
            else:
                d[col] = val
        rows.append(d)

    return rows, int(total_count)


def get_organization(org_id):
    """Fetch a single organization by ID. Returns a dict or None."""
    safe_id = sanitize_input(org_id)
    sql = f"SELECT * FROM {TABLE_NAME} WHERE organization_id = '{safe_id}'"
    df = spark.sql(sql).toPandas()
    if df.empty:
        return None
    row = df.iloc[0]
    d = {}
    for col in df.columns:
        val = row[col]
        if val is None or (hasattr(val, '__class__') and val.__class__.__name__ == 'NaTType'):
            d[col] = None
        elif hasattr(val, 'isoformat'):
            d[col] = val.isoformat()
        elif isinstance(val, float) and (val != val):
            d[col] = None
        else:
            d[col] = val
    return d


def update_organization(org_id, field_updates):
    """Update specific fields for an organization. Returns True on success."""
    if not field_updates:
        return False
    safe_id = sanitize_input(org_id)
    set_clauses = []

    for field, value in field_updates.items():
        if field not in EDITABLE_COLUMNS:
            continue
        if value is None or (isinstance(value, str) and value.strip() == ""):
            set_clauses.append(f"{field} = NULL")
        elif field == "bed_count":
            try:
                int_val = int(value)
                set_clauses.append(f"{field} = {int_val}")
            except (ValueError, TypeError):
                set_clauses.append(f"{field} = NULL")
        elif field == "is_subpart":
            bool_val = str(value).lower() in ("true", "1", "yes", "y")
            set_clauses.append(f"{field} = {str(bool_val).lower()}")
        else:
            safe_val = sanitize_input(value)
            set_clauses.append(f"{field} = '{safe_val}'")

    if not set_clauses:
        return False

    set_clauses.append("updated_date = current_timestamp()")
    set_str = ", ".join(set_clauses)
    sql = f"UPDATE {TABLE_NAME} SET {set_str} WHERE organization_id = '{safe_id}'"
    spark.sql(sql)
    return True


print("Database helper functions loaded.")

In [ ]:
# Cell 3: Flask App with HTML SPA

app = Flask(__name__)

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>MDM Organization Manager</title>
<style>
  * { box-sizing: border-box; margin: 0; padding: 0; }
  body { font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; background: #f0f2f5; color: #202124; }

  /* Header */
  .app-header { background: linear-gradient(135deg, #1a73e8, #0d47a1); color: #fff; padding: 20px 32px; display: flex; align-items: center; justify-content: space-between; box-shadow: 0 2px 8px rgba(0,0,0,.15); }
  .app-header h1 { font-size: 22px; font-weight: 600; }
  .app-header .subtitle { font-size: 13px; opacity: .85; margin-top: 2px; }

  /* Container */
  .container { max-width: 1400px; margin: 24px auto; padding: 0 24px; }

  /* Search */
  .search-bar { display: flex; gap: 10px; margin-bottom: 20px; align-items: center; }
  .search-bar input { flex: 1; max-width: 500px; padding: 10px 16px; border: 2px solid #dadce0; border-radius: 8px; font-size: 14px; outline: none; transition: border-color .2s; }
  .search-bar input:focus { border-color: #1a73e8; }
  .btn { padding: 10px 20px; border: none; border-radius: 8px; font-size: 14px; font-weight: 500; cursor: pointer; transition: all .2s; display: inline-flex; align-items: center; gap: 6px; }
  .btn-primary { background: #1a73e8; color: #fff; }
  .btn-primary:hover { background: #1557b0; }
  .btn-secondary { background: #e8eaed; color: #3c4043; }
  .btn-secondary:hover { background: #d2d5d9; }
  .btn-success { background: #137333; color: #fff; }
  .btn-success:hover { background: #0d5a27; }
  .btn-warning { background: #ea8600; color: #fff; }
  .btn-warning:hover { background: #c77200; }
  .btn-sm { padding: 6px 12px; font-size: 12px; border-radius: 6px; }

  /* Stats */
  .stats-bar { background: #fff; padding: 12px 20px; border-radius: 8px 8px 0 0; border: 1px solid #dadce0; border-bottom: none; color: #5f6368; font-size: 13px; display: flex; justify-content: space-between; align-items: center; }

  /* Table */
  .table-wrapper { overflow-x: auto; border: 1px solid #dadce0; border-radius: 0 0 8px 8px; background: #fff; }
  table { width: 100%; border-collapse: collapse; font-size: 13px; }
  thead th { background: #f8f9fa; color: #202124; padding: 12px 14px; text-align: left; font-weight: 600; border-bottom: 2px solid #dadce0; white-space: nowrap; position: sticky; top: 0; }
  tbody td { padding: 10px 14px; border-bottom: 1px solid #e8eaed; max-width: 220px; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
  tbody tr:nth-child(even) { background: #fafbfc; }
  tbody tr:hover { background: #e8f0fe; }
  .cell-id { font-family: 'Roboto Mono', monospace; font-size: 12px; color: #5f6368; }

  /* Status badges */
  .badge { padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: 600; display: inline-block; }
  .badge-active { background: #e6f4ea; color: #137333; }
  .badge-inactive { background: #fce8e6; color: #c5221f; }
  .badge-deactivated { background: #f1f3f4; color: #5f6368; }

  /* Edit button in row */
  .edit-row-btn { background: none; border: 1px solid #dadce0; border-radius: 6px; padding: 5px 10px; cursor: pointer; color: #1a73e8; font-size: 12px; transition: all .2s; display: inline-flex; align-items: center; gap: 4px; }
  .edit-row-btn:hover { background: #e8f0fe; border-color: #1a73e8; }

  /* Pagination */
  .pagination { display: flex; align-items: center; justify-content: center; gap: 12px; padding: 16px; }
  .pagination .btn { min-width: 100px; justify-content: center; }
  .page-info { font-size: 14px; color: #5f6368; min-width: 140px; text-align: center; }

  /* Highlight */
  mark { background: #fdd835; padding: 1px 3px; border-radius: 2px; }

  /* Modal overlay */
  .modal-overlay { display: none; position: fixed; top: 0; left: 0; width: 100%; height: 100%; background: rgba(0,0,0,.5); z-index: 1000; justify-content: center; align-items: flex-start; padding-top: 40px; overflow-y: auto; }
  .modal-overlay.active { display: flex; }
  .modal { background: #fff; border-radius: 12px; width: 680px; max-width: 95vw; box-shadow: 0 8px 32px rgba(0,0,0,.25); animation: slideDown .25s ease; }
  @keyframes slideDown { from { opacity: 0; transform: translateY(-20px); } to { opacity: 1; transform: translateY(0); } }
  .modal-header { background: linear-gradient(135deg, #1a73e8, #0d47a1); color: #fff; padding: 18px 24px; border-radius: 12px 12px 0 0; display: flex; justify-content: space-between; align-items: center; }
  .modal-header h2 { font-size: 17px; font-weight: 600; }
  .modal-close { background: none; border: none; color: #fff; font-size: 22px; cursor: pointer; padding: 4px 8px; border-radius: 4px; }
  .modal-close:hover { background: rgba(255,255,255,.2); }
  .modal-readonly { background: #f8f9fa; padding: 10px 24px; font-size: 12px; color: #5f6368; border-bottom: 1px solid #e0e0e0; }
  .modal-body { padding: 24px; }
  .form-group { margin-bottom: 16px; }
  .form-group label { display: block; font-size: 13px; font-weight: 500; color: #5f6368; margin-bottom: 4px; }
  .form-group input, .form-group select, .form-group textarea { width: 100%; padding: 9px 12px; border: 1px solid #dadce0; border-radius: 6px; font-size: 14px; outline: none; transition: border-color .2s; font-family: inherit; }
  .form-group input:focus, .form-group select:focus, .form-group textarea:focus { border-color: #1a73e8; box-shadow: 0 0 0 2px rgba(26,115,232,.15); }
  .form-group textarea { resize: vertical; min-height: 60px; }
  .form-row { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; }
  .modal-footer { padding: 16px 24px; border-top: 1px solid #e0e0e0; display: flex; justify-content: flex-end; gap: 10px; }

  /* Toast */
  .toast-container { position: fixed; top: 20px; right: 20px; z-index: 2000; display: flex; flex-direction: column; gap: 8px; }
  .toast { padding: 12px 20px; border-radius: 8px; color: #fff; font-size: 14px; font-weight: 500; box-shadow: 0 4px 12px rgba(0,0,0,.15); animation: fadeIn .3s ease; min-width: 280px; }
  .toast-success { background: #137333; }
  .toast-error { background: #c5221f; }
  .toast-info { background: #1a73e8; }
  @keyframes fadeIn { from { opacity: 0; transform: translateX(20px); } to { opacity: 1; transform: translateX(0); } }

  /* Loading */
  .loading { text-align: center; padding: 40px; color: #5f6368; }
  .spinner { display: inline-block; width: 24px; height: 24px; border: 3px solid #dadce0; border-top-color: #1a73e8; border-radius: 50%; animation: spin .8s linear infinite; margin-right: 8px; vertical-align: middle; }
  @keyframes spin { to { transform: rotate(360deg); } }

  /* No results */
  .no-results { text-align: center; padding: 60px 20px; color: #5f6368; }
  .no-results h3 { font-size: 18px; margin-bottom: 8px; color: #3c4043; }
</style>
</head>
<body>

<div class="app-header">
  <div>
    <h1>MDM Organization Manager</h1>
    <div class="subtitle">Table: mdm_genai_assets.demoui.orgtable</div>
  </div>
  <div style="font-size:13px;opacity:.8;" id="lastRefresh"></div>
</div>

<div class="container">
  <div class="search-bar">
    <input type="text" id="searchInput" placeholder="Search by organization name or ID..." autocomplete="off">
    <button class="btn btn-primary" onclick="doSearch()">&#128269; Search</button>
    <button class="btn btn-secondary" onclick="clearSearch()">Clear</button>
    <button class="btn btn-secondary" onclick="refreshData()">&#x21bb; Refresh</button>
  </div>

  <div class="stats-bar" id="statsBar">Ready to search...</div>

  <div class="table-wrapper">
    <table>
      <thead>
        <tr>
          <th style="width:40px">#</th>
          <th>Organization ID</th>
          <th>Organization Name</th>
          <th>Status</th>
          <th>Beds</th>
          <th>Source</th>
          <th>Parent Organization</th>
          <th>GenAI Name</th>
          <th>GenAI Specialty</th>
          <th style="width:80px">Actions</th>
        </tr>
      </thead>
      <tbody id="tableBody">
        <tr><td colspan="10" class="loading"><span class="spinner"></span> Loading data...</td></tr>
      </tbody>
    </table>
  </div>

  <div class="pagination" id="paginationBar">
    <button class="btn btn-secondary" id="prevBtn" onclick="prevPage()" disabled>&#9664; Previous</button>
    <span class="page-info" id="pageInfo">Page 1</span>
    <button class="btn btn-secondary" id="nextBtn" onclick="nextPage()">Next &#9654;</button>
  </div>
</div>

<!-- Edit Modal -->
<div class="modal-overlay" id="editModal">
  <div class="modal">
    <div class="modal-header">
      <h2 id="modalTitle">Edit Organization</h2>
      <button class="modal-close" onclick="closeModal()">&times;</button>
    </div>
    <div class="modal-readonly" id="modalReadonly"></div>
    <div class="modal-body">
      <input type="hidden" id="editOrgId">
      <div class="form-row">
        <div class="form-group">
          <label>Organization Name</label>
          <input type="text" id="editName">
        </div>
        <div class="form-group">
          <label>Status</label>
          <select id="editStatus">
            <option value="A">A - Active</option>
            <option value="I">I - Inactive</option>
            <option value="D">D - Deactivated</option>
          </select>
        </div>
      </div>
      <div class="form-row">
        <div class="form-group">
          <label>Bed Count</label>
          <input type="number" id="editBedCount">
        </div>
        <div class="form-group">
          <label>Is Subpart</label>
          <select id="editIsSubpart">
            <option value="">Unknown</option>
            <option value="true">Yes</option>
            <option value="false">No</option>
          </select>
        </div>
      </div>
      <div class="form-group">
        <label>Parent Organization Name</label>
        <input type="text" id="editParentOrg">
      </div>
      <div class="form-row">
        <div class="form-group">
          <label>GenAI Organization Name</label>
          <input type="text" id="editGenaiName">
        </div>
        <div class="form-group">
          <label>GenAI Primary Specialty</label>
          <input type="text" id="editGenaiSpecialty">
        </div>
      </div>
      <div class="form-group">
        <label>GenAI Citation</label>
        <textarea id="editGenaiCitation" rows="3"></textarea>
      </div>
    </div>
    <div class="modal-footer">
      <button class="btn btn-secondary" onclick="closeModal()">Cancel</button>
      <button class="btn btn-success" id="saveBtn" onclick="saveChanges()">&#10003; Save Changes</button>
    </div>
  </div>
</div>

<div class="toast-container" id="toastContainer"></div>

<script>
  let currentQuery = '';
  let currentPage = 0;
  let totalCount = 0;
  let pageSize = 25;
  let originalData = {};

  // Initialize
  document.addEventListener('DOMContentLoaded', () => {
    document.getElementById('searchInput').addEventListener('keydown', e => {
      if (e.key === 'Enter') doSearch();
    });
    fetchData();
  });

  function doSearch() {
    currentQuery = document.getElementById('searchInput').value.trim();
    currentPage = 0;
    fetchData();
  }

  function clearSearch() {
    document.getElementById('searchInput').value = '';
    currentQuery = '';
    currentPage = 0;
    fetchData();
  }

  function refreshData() { fetchData(); }
  function prevPage() { if (currentPage > 0) { currentPage--; fetchData(); } }
  function nextPage() { const tp = Math.max(1, Math.ceil(totalCount / pageSize)); if (currentPage < tp - 1) { currentPage++; fetchData(); } }

  function fetchData() {
    const tbody = document.getElementById('tableBody');
    tbody.innerHTML = '<tr><td colspan="10" class="loading"><span class="spinner"></span> Loading...</td></tr>';

    const params = new URLSearchParams({ q: currentQuery, page: currentPage, pageSize: pageSize });
    fetch('/api/organizations?' + params)
      .then(r => r.json())
      .then(data => {
        totalCount = data.totalCount;
        renderTable(data.rows, data.page, data.totalCount);
        updatePagination();
        updateStats();
        document.getElementById('lastRefresh').textContent = 'Last refreshed: ' + new Date().toLocaleTimeString();
      })
      .catch(err => {
        tbody.innerHTML = '<tr><td colspan="10" class="no-results"><h3>Error loading data</h3><p>' + err.message + '</p></td></tr>';
        showToast('Error fetching data: ' + err.message, 'error');
      });
  }

  function renderTable(rows, page, total) {
    const tbody = document.getElementById('tableBody');
    if (!rows || rows.length === 0) {
      tbody.innerHTML = '<tr><td colspan="10" class="no-results"><h3>No organizations found</h3><p>Try a different search term.</p></td></tr>';
      return;
    }
    const start = page * pageSize + 1;
    let html = '';
    rows.forEach((row, i) => {
      const num = start + i;
      const orgId = esc(row.organization_id || '');
      const orgName = esc(row.organization_name || '');
      const status = row.organization_status || '';
      const beds = row.bed_count != null ? row.bed_count : '-';
      const source = esc(row.source_system || '-');
      const parent = esc(row.parent_organization_name || '-');
      const genaiName = esc(row.genai_organization_name || '-');
      const genaiSpec = esc(row.genai_primary_specialty || '-');

      html += '<tr>';
      html += '<td style="color:#9aa0a6">' + num + '</td>';
      html += '<td class="cell-id">' + highlight(orgId) + '</td>';
      html += '<td>' + highlight(orgName) + '</td>';
      html += '<td>' + statusBadge(status) + '</td>';
      html += '<td>' + beds + '</td>';
      html += '<td>' + source + '</td>';
      html += '<td title="' + parent + '">' + parent + '</td>';
      html += '<td title="' + genaiName + '">' + genaiName + '</td>';
      html += '<td title="' + genaiSpec + '">' + genaiSpec + '</td>';
      html += '<td><button class="edit-row-btn" onclick="openEditModal(\\'' + escAttr(row.organization_id) + '\\')">&#9998; Edit</button></td>';
      html += '</tr>';
    });
    tbody.innerHTML = html;
  }

  function esc(str) {
    const d = document.createElement('div');
    d.textContent = str;
    return d.innerHTML;
  }

  function escAttr(str) {
    return String(str || '').replace(/\\\\/g, '\\\\\\\\').replace(/'/g, "\\\\'");
  }

  function highlight(text) {
    if (!currentQuery || !text) return text;
    const tokens = currentQuery.trim().split(/\\s+/);
    let result = text;
    tokens.forEach(token => {
      if (token) {
        const re = new RegExp('(' + token.replace(/[.*+?^${}()|[\\]\\\\]/g, '\\\\$&') + ')', 'gi');
        result = result.replace(re, '<mark>$1</mark>');
      }
    });
    return result;
  }

  function statusBadge(s) {
    if (!s) return '';
    const u = s.toUpperCase();
    if (u === 'A') return '<span class="badge badge-active">Active</span>';
    if (u === 'I') return '<span class="badge badge-inactive">Inactive</span>';
    if (u === 'D') return '<span class="badge badge-deactivated">Deactivated</span>';
    return '<span class="badge badge-deactivated">' + esc(s) + '</span>';
  }

  function updatePagination() {
    const tp = Math.max(1, Math.ceil(totalCount / pageSize));
    document.getElementById('prevBtn').disabled = currentPage <= 0;
    document.getElementById('nextBtn').disabled = currentPage >= tp - 1;
    document.getElementById('pageInfo').textContent = 'Page ' + (currentPage + 1) + ' of ' + tp;
  }

  function updateStats() {
    const start = currentPage * pageSize + 1;
    const end = Math.min(start + pageSize - 1, totalCount);
    const searchInfo = currentQuery ? ' for "' + esc(currentQuery) + '"' : '';
    document.getElementById('statsBar').innerHTML =
      'Showing ' + (totalCount > 0 ? start : 0) + '-' + (totalCount > 0 ? end : 0) +
      ' of ' + totalCount.toLocaleString() + ' results' + searchInfo;
  }

  // Edit Modal
  function openEditModal(orgId) {
    showToast('Loading record...', 'info');
    fetch('/api/organizations/' + encodeURIComponent(orgId))
      .then(r => r.json())
      .then(data => {
        if (data.error) { showToast(data.error, 'error'); return; }
        originalData = data;
        document.getElementById('editOrgId').value = data.organization_id || '';
        document.getElementById('modalTitle').textContent = 'Edit: ' + (data.organization_id || '');
        document.getElementById('modalReadonly').innerHTML =
          '<b>Source:</b> ' + esc(data.source_system || '-') +
          ' &nbsp;|&nbsp; <b>Source Record ID:</b> ' + esc(data.source_record_id || '-') +
          ' &nbsp;|&nbsp; <b>Created:</b> ' + esc(data.created_date || '-') +
          ' &nbsp;|&nbsp; <b>Last Updated:</b> ' + esc(data.updated_date || '-') +
          ' &nbsp;|&nbsp; <b>Enumeration Date:</b> ' + esc(data.enumeration_date || '-');

        document.getElementById('editName').value = data.organization_name || '';
        document.getElementById('editStatus').value = data.organization_status || 'A';
        document.getElementById('editBedCount').value = data.bed_count != null ? data.bed_count : '';
        document.getElementById('editIsSubpart').value = data.is_subpart === true ? 'true' : data.is_subpart === false ? 'false' : '';
        document.getElementById('editParentOrg').value = data.parent_organization_name || '';
        document.getElementById('editGenaiName').value = data.genai_organization_name || '';
        document.getElementById('editGenaiSpecialty').value = data.genai_primary_specialty || '';
        document.getElementById('editGenaiCitation').value = data.genai_citation || '';

        document.getElementById('editModal').classList.add('active');
      })
      .catch(err => showToast('Error loading record: ' + err.message, 'error'));
  }

  function closeModal() {
    document.getElementById('editModal').classList.remove('active');
  }

  function saveChanges() {
    const orgId = document.getElementById('editOrgId').value;
    const updates = {};

    const name = document.getElementById('editName').value;
    if (name !== (originalData.organization_name || '')) updates.organization_name = name;

    const status = document.getElementById('editStatus').value;
    if (status !== (originalData.organization_status || '')) updates.organization_status = status;

    const beds = document.getElementById('editBedCount').value;
    const origBeds = originalData.bed_count != null ? String(originalData.bed_count) : '';
    if (beds !== origBeds) updates.bed_count = beds || null;

    const subpart = document.getElementById('editIsSubpart').value;
    const origSubpart = originalData.is_subpart === true ? 'true' : originalData.is_subpart === false ? 'false' : '';
    if (subpart !== origSubpart) updates.is_subpart = subpart || null;

    const parent = document.getElementById('editParentOrg').value;
    if (parent !== (originalData.parent_organization_name || '')) updates.parent_organization_name = parent;

    const gName = document.getElementById('editGenaiName').value;
    if (gName !== (originalData.genai_organization_name || '')) updates.genai_organization_name = gName;

    const gSpec = document.getElementById('editGenaiSpecialty').value;
    if (gSpec !== (originalData.genai_primary_specialty || '')) updates.genai_primary_specialty = gSpec;

    const gCite = document.getElementById('editGenaiCitation').value;
    if (gCite !== (originalData.genai_citation || '')) updates.genai_citation = gCite;

    if (Object.keys(updates).length === 0) {
      showToast('No changes detected.', 'info');
      return;
    }

    const saveBtn = document.getElementById('saveBtn');
    saveBtn.disabled = true;
    saveBtn.innerHTML = '<span class="spinner" style="width:16px;height:16px;border-width:2px;margin-right:6px"></span> Saving...';

    fetch('/api/organizations/' + encodeURIComponent(orgId), {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify(updates)
    })
    .then(r => r.json())
    .then(data => {
      saveBtn.disabled = false;
      saveBtn.innerHTML = '&#10003; Save Changes';
      if (data.success) {
        const fields = Object.keys(updates).join(', ');
        showToast('Successfully updated ' + fields + ' for ' + orgId, 'success');
        closeModal();
        fetchData();  // Refresh table to show latest data
      } else {
        showToast('Update failed: ' + (data.error || 'Unknown error'), 'error');
      }
    })
    .catch(err => {
      saveBtn.disabled = false;
      saveBtn.innerHTML = '&#10003; Save Changes';
      showToast('Error saving: ' + err.message, 'error');
    });
  }

  // Toast notifications
  function showToast(message, type) {
    const container = document.getElementById('toastContainer');
    const toast = document.createElement('div');
    toast.className = 'toast toast-' + (type || 'info');
    toast.textContent = message;
    container.appendChild(toast);
    setTimeout(() => { toast.style.opacity = '0'; toast.style.transition = 'opacity .3s'; setTimeout(() => toast.remove(), 300); }, 4000);
  }

  // Close modal on overlay click
  document.getElementById('editModal').addEventListener('click', e => {
    if (e.target === document.getElementById('editModal')) closeModal();
  });
  // Close modal on Escape
  document.addEventListener('keydown', e => { if (e.key === 'Escape') closeModal(); });
</script>
</body>
</html>"""


# --- Flask Routes ---

@app.route("/")
def index():
    return Response(HTML_TEMPLATE, mimetype="text/html")


@app.route("/api/organizations")
def api_search():
    query = request.args.get("q", "").strip()
    page = int(request.args.get("page", 0))
    page_size_param = int(request.args.get("pageSize", PAGE_SIZE))
    try:
        rows, total = search_organizations(query, page, page_size_param)
        return jsonify({"rows": rows, "totalCount": total, "page": page, "pageSize": page_size_param})
    except Exception as e:
        return jsonify({"error": str(e), "rows": [], "totalCount": 0, "page": 0, "pageSize": page_size_param}), 500


@app.route("/api/organizations/<org_id>", methods=["GET"])
def api_get_org(org_id):
    try:
        org = get_organization(org_id)
        if org is None:
            return jsonify({"error": f"Organization {org_id} not found"}), 404
        return jsonify(org)
    except Exception as e:
        return jsonify({"error": str(e)}), 500


@app.route("/api/organizations/<org_id>", methods=["POST"])
def api_update_org(org_id):
    try:
        updates = request.get_json()
        if not updates:
            return jsonify({"success": False, "error": "No update data provided"})
        success = update_organization(org_id, updates)
        if success:
            updated_org = get_organization(org_id)
            return jsonify({"success": True, "organization": updated_org})
        else:
            return jsonify({"success": False, "error": "No valid fields to update"})
    except Exception as e:
        return jsonify({"success": False, "error": str(e)}), 500


print("Flask app defined with routes: /, /api/organizations, /api/organizations/<id>")

In [ ]:
# Cell 5: Start Flask Server & Display Link

import threading
import os
from IPython.display import display, HTML

# Databricks workspace configuration (fallbacks)
DATABRICKS_HOST = "dbc-bb5a5175-0b04.cloud.databricks.com"
DATABRICKS_ORG_ID = "2960871271454688"

def start_server():
    """Start Flask in a background thread."""
    app.run(host="0.0.0.0", port=FLASK_PORT, debug=False, use_reloader=False)

# Start the server
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()

# Try multiple methods to get the cluster ID and build the proxy URL
host = DATABRICKS_HOST
org_id = DATABRICKS_ORG_ID
cluster_id = None

# Method 1: dbutils context
try:
    context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
    host = context.browserHostName().getOrElse(None) or host
    org_id = context.orgId().getOrElse(None) or org_id
    cluster_id = context.clusterId().getOrElse(None)
except Exception:
    pass

# Method 2: Spark conf
if not cluster_id:
    try:
        cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
    except Exception:
        pass

# Method 3: Environment variable
if not cluster_id:
    cluster_id = os.environ.get("DB_CLUSTER_ID") or os.environ.get("DATABRICKS_CLUSTER_ID")

# Method 4: Spark conf for org ID if still missing
if org_id == DATABRICKS_ORG_ID:
    try:
        org_id = spark.conf.get("spark.databricks.clusterUsageTags.orgId") or org_id
    except Exception:
        pass

# Debug: print what we found (helpful for troubleshooting)
print(f"Host: {host}")
print(f"Org ID: {org_id}")
print(f"Cluster ID: {cluster_id}")
print(f"Port: {FLASK_PORT}")

if cluster_id:
    proxy_url = f"https://{host}/driver-proxy/o/{org_id}/{cluster_id}/{FLASK_PORT}/"
    display(HTML(f"""
    <div style="padding: 20px; background: #e8f5e9; border-radius: 8px; border: 1px solid #a5d6a7; margin: 10px 0;">
        <h3 style="color: #2e7d32; margin: 0 0 10px 0;">&#9989; MDM Organization Manager is running!</h3>
        <p style="margin: 0 0 12px 0;">Click the button below to open the UI in a new browser tab:</p>
        <a href="{proxy_url}" target="_blank"
           style="display: inline-block; padding: 12px 28px; background: #1a73e8; color: white;
                  text-decoration: none; border-radius: 8px; font-weight: 600; font-size: 15px;
                  box-shadow: 0 2px 8px rgba(26,115,232,.3);">
            &#128640; Open MDM Organization Manager
        </a>
        <p style="margin: 12px 0 0 0; font-size: 12px; color: #5f6368;">
            URL: <code>{proxy_url}</code>
        </p>
    </div>
    """))
else:
    # Could not detect cluster ID - show manual fallback
    display(HTML(f"""
    <div style="padding: 20px; background: #fff3e0; border-radius: 8px; border: 1px solid #ffe0b2; margin: 10px 0;">
        <h3 style="color: #e65100; margin: 0 0 10px 0;">&#9989; Server is running on port {FLASK_PORT}</h3>
        <p style="margin: 0 0 8px 0;">Could not auto-detect cluster ID. To get your Cluster ID:</p>
        <ol style="font-size: 13px; margin: 8px 0; padding-left: 20px;">
            <li>Go to <b>Compute</b> in the left sidebar</li>
            <li>Click on your running cluster</li>
            <li>Copy the cluster ID from the URL (e.g. <code>0327-123456-abcdef</code>)</li>
            <li>Your full URL will be:</li>
        </ol>
        <code style="background: #f5f5f5; padding: 8px 12px; border-radius: 4px; display: block; margin: 8px 0; font-size: 13px;">
            https://{host}/driver-proxy/o/{org_id}/&lt;YOUR_CLUSTER_ID&gt;/{FLASK_PORT}/
        </code>
    </div>
    """))

print(f"Server started on port {FLASK_PORT}")